In [32]:
import os
from collections import defaultdict
import torch
import torch.nn.functional as F
import random
import math
import numpy as np
import scipy.sparse as sp
from sklearn import metrics
from munkres import Munkres                       
import skfuzzy as fuzz
from sklearn.metrics import adjusted_rand_score as ari_score
from sklearn.metrics.cluster import normalized_mutual_info_score as nmi_score

In [41]:
# Create an undirected graph with 5 nodes and nodes 0,1,2 are connected to each other, and nodes 3,4 are connected to each other
adjacency_matrix = torch.tensor([[0, 1, 1, 0, 0],
                                  [1, 0, 1, 0, 0],
                                  [1, 1, 0, 0, 0],
                                  [0, 0, 0, 0, 1],
                                  [0, 0, 0, 1, 0]])
# Create a feature matrix for the 5 nodes, with 2 features each
feature_matrix = torch.tensor([[0.8, 0.3],
                                [0.6, 0.1],  
                                [0.7, 0.2],  
                                [0.1, 0.9],  
                                [0.2, 0.8]])

# Create labels into a list of sets format
labels = [{0, 1, 2}, {3, 4}]

In [31]:
def eva(C_star, C_hat, num_nodes):
    """
    Calculates the Overlapping Normalized Mutual Information (ONMI)
    as formulated by Lancichinetti, Fortunato, and Kertesz (LFK).
    """
    def entropy(p_list):
        """Calculates Shannon entropy for a probability distribution."""
        return -sum([p * math.log2(p) for p in p_list if p > 0])

    def calc_conditional_entropy(c1, c2, N):
        """Calculates H(X), H(Y), and H(X|Y) for two sets of nodes."""
        intersect = len(c1.intersection(c2))
        c1_only = len(c1) - intersect
        c2_only = len(c2) - intersect
        neither = N - (intersect + c1_only + c2_only)
        
        # Joint distribution matrix P(X, Y)
        p_11 = intersect / N
        p_10 = c1_only / N
        p_01 = c2_only / N
        p_00 = neither / N
        
        # Marginal distributions P(X) and P(Y)
        p_x1 = len(c1) / N
        p_x0 = 1.0 - p_x1
        p_y1 = len(c2) / N
        p_y0 = 1.0 - p_y1
        
        H_X = entropy([p_x0, p_x1])
        H_Y = entropy([p_y0, p_y1])
        H_XY = entropy([p_00, p_01, p_10, p_11])
        
        # H(X | Y) = H(X, Y) - H(Y)
        H_X_given_Y = H_XY - H_Y
        return H_X, H_Y, H_X_given_Y

    if not C_star or not C_hat:
        return 0.0

    # 1. For each true community, find the best matching predicted community
    H_X_given_Y_norm = []
    for c_x in C_star:
        H_X, _, _ = calc_conditional_entropy(c_x, set(), num_nodes)
        if H_X == 0:
            continue # Skip edge-case communities that contain ALL or ZERO nodes
            
        min_H_X_given_Y = float('inf')
        for c_y in C_hat:
            _, _, H_X_given_Y = calc_conditional_entropy(c_x, c_y, num_nodes)
            min_H_X_given_Y = min(min_H_X_given_Y, H_X_given_Y)
            
        # Normalize and cap at 1.0 (LFK standard for finite sample variations)
        H_X_given_Y_norm.append(min(1.0, min_H_X_given_Y / H_X))

    # 2. For each predicted community, find the best matching true community
    H_Y_given_X_norm = []
    for c_y in C_hat:
        H_Y, _, _ = calc_conditional_entropy(c_y, set(), num_nodes)
        if H_Y == 0:
            continue
            
        min_H_Y_given_X = float('inf')
        for c_x in C_star:
            # Notice the symmetry flip: we want H(Y|X) here, which is H(X,Y) - H(X).
            # Passing c_y first to our helper function returns H(c_y | c_x).
            _, _, H_Y_given_X = calc_conditional_entropy(c_y, c_x, num_nodes)
            min_H_Y_given_X = min(min_H_Y_given_X, H_Y_given_X)
            
        H_Y_given_X_norm.append(min(1.0, min_H_Y_given_X / H_Y))

    # 3. Average the normalized conditional entropies
    avg_H_X_given_Y = np.mean(H_X_given_Y_norm) if H_X_given_Y_norm else 1.0
    avg_H_Y_given_X = np.mean(H_Y_given_X_norm) if H_Y_given_X_norm else 1.0

    # 4. Final ONMI calculation
    onmi = 1.0 - 0.5 * (avg_H_X_given_Y + avg_H_Y_given_X)
    
    return max(0.0, onmi)

In [ ]:
# test this clustering function
def clustering(feature, true_labels, cluster_num, device):
    # 1. Prepare data for fuzzy c-means (requires CPU numpy array)
    # feature shape: (N, D) -> skfuzzy requires (D, N)
    feature_np = feature.detach().cpu().numpy().T  
    
    # 2. Run Fuzzy C-Means
    cntr, u, _, _, _, _, _ = fuzz.cluster.cmeans(
        data=feature_np, 
        c=cluster_num, 
        m=2.0, 
        error=0.005, 
        maxiter=1000, 
        init=None
    )

    u = u.T  # Transpose back to (n_samples, n_clusters)

    # 3. Move results back to PyTorch and correct device
    centers = torch.from_numpy(cntr).float().to(device)
    predict_labels_soft = torch.from_numpy(u).float().to(device)

    '''
    '''
    # Convert U to hard labels using argmax for dominant cluster assignment
    predict_labels_hard = torch.argmax(predict_labels_soft, dim=1)

    # Convert predict_labels_hard into a list of sets format (C_hat)
    predict_labels_hard_np = predict_labels_hard.cpu().numpy()
    C_hat = []
    for c in range(cluster_num):
        member_nodes = set(np.where(predict_labels_hard_np == c)[0].tolist())
        C_hat.append(member_nodes)

    # # 4. Process labels
    # u_max = predict_labels_soft.max(dim=1, keepdim=True).values
    # u_norm = predict_labels_soft / (u_max + 1e-8)

    # predict_labels_matrix = (u_norm > 0.5).float()
    
    # # Convert the (N, K) binary matrix into a list of sets format (C_hat)
    # predict_labels_matrix_np = predict_labels_matrix.cpu().numpy()
    # C_hat = []
    
    # for c in range(cluster_num):
    #     # Find all row indices (nodes) where column 'c' is 1
    #     member_nodes = set(np.where(predict_labels_matrix_np[:, c] == 1.0)[0].tolist())
    #     C_hat.append(member_nodes)
        
    # Calculate NMI using the custom BigClam evaluation function
    # true_labels must be passed in as C_star (list of sets)
    nmi = eva(true_labels, C_hat, num_nodes=feature.shape[0])

    # 5. Compute distances
    # Ensure 'feature' is on the same device as 'centers'
    dis = torch.cdist(feature.to(device), centers).pow(2)
    
    return 100 * nmi, predict_labels_soft, predict_labels_hard, centers, dis

In [30]:
# test the clustering function with the toy data
nmi, predict_labels_soft, predict_labels_hard, centers, dis = clustering(feature_matrix, labels, cluster_num=2, device='cuda')
print("Soft Membership Matrix (U):")
print(predict_labels_soft.cpu().numpy())

print("Hard Predicted Labels:")
print(predict_labels_hard.cpu().numpy())

print("Cluster Centers:")
print(centers.cpu().numpy())

print("Distance Matrix:")
print(dis.cpu().numpy())

print(f"NMI: {nmi:.4f}")

Soft Membership Matrix (U):
[[9.7307956e-01 2.6920412e-02]
 [9.7453874e-01 2.5461281e-02]
 [1.0000000e+00 2.2160208e-08]
 [5.9271296e-03 9.9407285e-01]
 [8.0216471e-03 9.9197835e-01]]
Hard Predicted Labels:
[0 0 0 1 1]
Cluster Centers:
[[0.6998928  0.19993247]
 [0.1503069  0.84963024]]
Distance Matrix:
[[2.0034961e-02 7.2419453e-01]
 [1.9965069e-02 7.6416934e-01]
 [1.6045808e-08 7.2418195e-01]
 [8.4996593e-01 5.0678942e-03]
 [6.0997385e-01 4.9325638e-03]]
NMI: 100.0000


In [43]:
from torch import device
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# create toy state that is somewhat similar to the feature matrix but with some noise
state = (feature_matrix + 0.1 * torch.randn_like(feature_matrix)).to(device)

#after getting the state from the encoder, and after the agent has taken its action, we can perform clustering on the state to get the cluster centers 
# and the predicted labels. Cluster_num is chosen by the agent.

nmi, predict_labels_soft, predict_labels_hard, centers, dis = clustering(state, labels, cluster_num=2, device='cuda')
predict_labels = torch.tensor(predict_labels_hard).long().to(device)

one_hot = F.one_hot(predict_labels, num_classes=2).float()

numerator = torch.mm(one_hot.T, state)
denominator = one_hot.sum(dim=0, keepdim=True).T

cluster_state = numerator / (denominator + 1e-8)

C:\Users\lbvu3\AppData\Local\Temp\ipykernel_13212\3294587073.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  predict_labels = torch.tensor(predict_labels_hard).long().to(device)


In [44]:
print("one_hot:")
print(one_hot.cpu().numpy())

print("State:")
print(state.cpu().numpy())

print("Numerator (sum of states in each cluster):")
print(numerator.cpu().numpy())

print("Denominator (number of nodes in each cluster):")
print(denominator.cpu().numpy())

print("Cluster State (centers):")
print(cluster_state.cpu().numpy())

one_hot:
[[1. 0.]
 [1. 0.]
 [1. 0.]
 [0. 1.]
 [0. 1.]]
State:
[[0.73200476 0.33854035]
 [0.63318634 0.12145639]
 [0.66293335 0.20137396]
 [0.12483013 0.9900292 ]
 [0.19092476 0.9551547 ]]
Numerator (sum of states in each cluster):
[[2.0281243 0.6613707]
 [0.3157549 1.945184 ]]
Denominator (number of nodes in each cluster):
[[3.]
 [2.]]
Cluster State (centers):
[[0.6760414  0.2204569 ]
 [0.15787745 0.972592  ]]


In [48]:
dis = (state.unsqueeze(1) - centers.unsqueeze(0)).pow(2).sum(-1) + 1e-8
print("Distance Matrix:")
print(dis.cpu().numpy())

Distance Matrix:
[[1.7359518e-02 7.3114622e-01]
 [1.1405781e-02 9.4979042e-01]
 [4.8747921e-04 8.4932423e-01]
 [8.9702010e-01 1.4162682e-03]
 [7.7606130e-01 1.3762875e-03]]


In [ ]:
def eva(C_star, C_hat, num_nodes):
    """
    Calculates ONMI, Average F1 Score, and Pairwise Accuracy 
    for overlapping communities.
    """
    # ==========================================
    # Helper Functions
    # ==========================================
    def entropy(p_list):
        return -sum([p * math.log2(p) for p in p_list if p > 0])

    def calc_conditional_entropy(c1, c2, N):
        intersect = len(c1.intersection(c2))
        c1_only = len(c1) - intersect
        c2_only = len(c2) - intersect
        neither = N - (intersect + c1_only + c2_only)
        
        p_11 = intersect / N
        p_10 = c1_only / N
        p_01 = c2_only / N
        p_00 = neither / N
        
        p_x1 = len(c1) / N
        p_x0 = 1.0 - p_x1
        p_y1 = len(c2) / N
        p_y0 = 1.0 - p_y1
        
        H_X = entropy([p_x0, p_x1])
        H_Y = entropy([p_y0, p_y1])
        H_XY = entropy([p_00, p_01, p_10, p_11])
        
        H_X_given_Y = H_XY - H_Y
        return H_X, H_Y, H_X_given_Y

    def calc_f1(set_a, set_b):
        """Helper to calculate F1 score between two sets of nodes."""
        if not set_a or not set_b:
            return 0.0
        intersect = len(set_a.intersection(set_b))
        return (2.0 * intersect) / (len(set_a) + len(set_b))

    # Base check
    if not C_star or not C_hat:
        return 0.0, 0.0, 0.0

    # ==========================================
    # 1. ONMI Calculation
    # ==========================================
    H_X_given_Y_norm = []
    for c_x in C_star:
        H_X, _, _ = calc_conditional_entropy(c_x, set(), num_nodes)
        if H_X == 0:
            continue
            
        min_H_X_given_Y = float('inf')
        for c_y in C_hat:
            _, _, H_X_given_Y = calc_conditional_entropy(c_x, c_y, num_nodes)
            min_H_X_given_Y = min(min_H_X_given_Y, H_X_given_Y)
            
        H_X_given_Y_norm.append(min(1.0, min_H_X_given_Y / H_X))

    H_Y_given_X_norm = []
    for c_y in C_hat:
        H_Y, _, _ = calc_conditional_entropy(c_y, set(), num_nodes)
        if H_Y == 0:
            continue
            
        min_H_Y_given_X = float('inf')
        for c_x in C_star:
            _, _, H_Y_given_X = calc_conditional_entropy(c_y, c_x, num_nodes)
            min_H_Y_given_X = min(min_H_Y_given_X, H_Y_given_X)
            
        H_Y_given_X_norm.append(min(1.0, min_H_Y_given_X / H_Y))

    avg_H_X_given_Y = np.mean(H_X_given_Y_norm) if H_X_given_Y_norm else 1.0
    avg_H_Y_given_X = np.mean(H_Y_given_X_norm) if H_Y_given_X_norm else 1.0
    
    onmi = max(0.0, 1.0 - 0.5 * (avg_H_X_given_Y + avg_H_Y_given_X))

    # ==========================================
    # 2. Average F1 Score Calculation
    # ==========================================
    f1_star_to_hat = []
    for c_x in C_star:
        best_f1 = max([calc_f1(c_x, c_y) for c_y in C_hat])
        f1_star_to_hat.append(best_f1)
        
    f1_hat_to_star = []
    for c_y in C_hat:
        best_f1 = max([calc_f1(c_y, c_x) for c_x in C_star])
        f1_hat_to_star.append(best_f1)
        
    avg_f1 = 0.5 * (np.mean(f1_star_to_hat) + np.mean(f1_hat_to_star))

    # ==========================================
    # 3. Pairwise Accuracy Calculation
    # ==========================================
    shared_true = np.zeros((num_nodes, num_nodes), dtype=np.int32)
    shared_pred = np.zeros((num_nodes, num_nodes), dtype=np.int32)
    
    for c in C_star:
        nodes = list(c)
        for i in range(len(nodes)):
            for j in range(i, len(nodes)): 
                shared_true[nodes[i], nodes[j]] += 1
                if i != j: shared_true[nodes[j], nodes[i]] += 1
                
    for c in C_hat:
        nodes = list(c)
        for i in range(len(nodes)):
            for j in range(i, len(nodes)):
                shared_pred[nodes[i], nodes[j]] += 1
                if i != j: shared_pred[nodes[j], nodes[i]] += 1
                
    # Fraction of pairs where the number of shared communities matches exactly
    accuracy = np.mean(shared_true == shared_pred)

    return onmi, avg_f1, accuracy